<a href="https://colab.research.google.com/github/AngelGarcia0905/IA/blob/main/Actividad_8_Redes_Neuronales_Convolucionales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. IMPORTACIÓN DE LIBRERÍAS Y DIAGNÓSTICO
# -----------------------------------------
# Se cargan todas las herramientas de visión artificial y diagnóstico de hardware.

import platform # Para identificar el sistema operativo.
import psutil   # Para estadísticas de memoria RAM y CPU.
import os       # Para interactuar con el entorno de Google Colab.
import numpy as np # Para manejo de matrices y arreglos.
import matplotlib.pyplot as plt # Para graficar imágenes y resultados.
import tensorflow as tf # Librería principal de redes neuronales.
from tensorflow.keras import datasets, layers, models # Módulos específicos de Keras.

def get_system_info():
    print(f"{'='*20} DIAGNÓSTICO DEL SISTEMA {'='*20}")
    print(f"Sistema: {platform.system()} | RAM: {round(psutil.virtual_memory().total / (1024**3), 2)} GB")
    # Es altamente recomendado usar GPU para Redes Convolucionales.
    gpu = tf.config.list_physical_devices('GPU')
    print(f"GPU Aceleradora: {'DISPONIBLE' if gpu else 'NO DETECTADA (Se usará CPU)'}")

get_system_info() # Ejecución del diagnóstico.

==================== DIAGNÓSTICO DEL SISTEMA ====================
Sistema: Linux | RAM: 12.67 GB
GPU Aceleradora: DISPONIBLE


In [2]:
# 2. CARGA DE DATOS
# -----------------
# Carga de imágenes de dígitos escritos a mano del 0 al 9.

# Descarga y división automática de datos de entrenamiento y prueba.
(train_images, train_labels), (test_images, test_labels) = datasets.mnist.load_data()

print(f"Imágenes de entrenamiento: {train_images.shape[0]}") # 60,000 muestras.
print(f"Imágenes de prueba: {test_images.shape[0]}")        # 10,000 muestras.

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Imágenes de entrenamiento: 60000
Imágenes de prueba: 10000


In [3]:
# 3. PREPROCESAMIENTO PARA CNN
# ----------------------------
# A diferencia de la red anterior, aquí NO aplanamos la imagen al inicio;
# la CNN necesita la estructura de matriz (28, 28, 1).

# Paso 1: Redimensionar para incluir el canal de color (1 para escala de grises).
train_images = train_images.reshape((60000, 28, 28, 1))
test_images = test_images.reshape((10000, 28, 28, 1))

# Paso 2: Normalizar los píxeles para que estén en el rango [0, 1].
train_images, test_images = train_images / 255.0, test_images / 255.0

print("Imágenes redimensionadas y normalizadas correctamente.")

Imágenes redimensionadas y normalizadas correctamente.


In [4]:
# 4. DISEÑO DE LA ARQUITECTURA CNN
# --------------------------------
# Proponemos una red con capas de convolución para extraer características.

model = models.Sequential([
    # Primera capa convolucional: Detecta 32 patrones básicos.
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    # Capa de agrupación: Reduce el tamaño de la imagen y resalta lo importante.
    layers.MaxPooling2D((2, 2)),

    # Segunda capa convolucional: Detecta 64 patrones más complejos.
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Aplanado: Convierte el mapa de características en un vector para la capa densa.
    layers.Flatten(),
    # Capa densa (completamente conectada).
    layers.Dense(64, activation='relu'),
    # Capa de salida: 10 neuronas con activacion Softmax para clasificación.
    layers.Dense(10, activation='softmax')
])

# Configuración del entrenamiento.
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary() # Muestra el mapa de capas y parámetros.

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       102,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 121,930 (476.29 KB)

 Trainable params: 121,930 (476.29 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# 5. ENTRENAMIENTO (FIT)
# ----------------------
# Ajuste de pesos de la CNN con los datos de entrenamiento.

print("Iniciando entrenamiento...")
# Usaremos 5 épocas para observar la mejora rápida de la CNN.
history = model.fit(train_images, train_labels, epochs=5, validation_split=0.1)

Iniciando entrenamiento...
Epoch 1/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - accuracy: 0.9552 - loss: 0.1509 - val_accuracy: 0.9818 - val_loss: 0.0593
Epoch 2/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9842 - loss: 0.0498 - val_accuracy: 0.9868 - val_loss: 0.0438
Epoch 3/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9891 - loss: 0.0350 - val_accuracy: 0.9890 - val_loss: 0.0377
Epoch 4/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9912 - loss: 0.0259 - val_accuracy: 0.9913 - val_loss: 0.0307
Epoch 5/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9937 - loss: 0.0198 - val_accuracy: 0.9907 - val_loss: 0.0375


In [6]:
# 6. PRUEBA Y PREDICCIÓN
# ----------------------
# Evaluación final y muestra de resultados reales.

# Evaluación del rendimiento con datos desconocidos.
test_loss, test_acc = model.evaluate(test_images, test_labels, verbose=2)
print(f"\nExactitud en prueba: {test_acc*100:.2f}%")

# Obtener predicciones para el conjunto de prueba.
predicciones_prob = model.predict(test_images, verbose=0)
predicciones = np.argmax(predicciones_prob, axis=1)

print(f"\n{'='*10} 5 PREDICCIONES REALIZADAS {'='*10}")
for i in range(5):
    print(f"Muestra {i+1} -> Real: {test_labels[i]} | Predicción: {predicciones[i]}")

313/313 - 1s - 2ms/step - accuracy: 0.9905 - loss: 0.0289

Exactitud en prueba: 99.05%

========== 5 PREDICCIONES REALIZADAS ==========
Muestra 1 -> Real: 7 | Predicción: 7
Muestra 2 -> Real: 2 | Predicción: 2
Muestra 3 -> Real: 1 | Predicción: 1
Muestra 4 -> Real: 0 | Predicción: 0
Muestra 5 -> Real: 4 | Predicción: 4
